# LabJack U3 — Trigger Test

Wiring: **FIO1 (green) → LED → GND (red)**

FIO1 = bit1, so to turn it on send value `2` (binary `0010`)

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import u3

d = u3.U3()
print(d.configU3())

# Set all FIO as digital I/O
d.configIO(FIOAnalog=0, EIOAnalog=0)

# Set FIO0-FIO3 as output (4-bit mask = 0x0F)
d.getFeedback(u3.PortDirWrite(Direction=[0x0F, 0, 0], WriteMask=[0x0F, 0, 0]))

# Reset to 0
d.getFeedback(u3.PortStateWrite(State=[0, 0, 0], WriteMask=[0x0F, 0, 0]))

print('LabJack U3 connected, FIO0-FIO3 initialised')

{'FirmwareVersion': '1.46', 'BootloaderVersion': '0.27', 'HardwareVersion': '1.30', 'SerialNumber': 320104792, 'ProductID': 3, 'LocalID': 1, 'TimerCounterMask': 64, 'FIOAnalog': 0, 'FIODirection': 0, 'FIOState': 0, 'EIOAnalog': 0, 'EIODirection': 0, 'EIOState': 0, 'CIODirection': 0, 'CIOState': 0, 'DAC1Enable': 1, 'DAC0': 0, 'DAC1': 0, 'TimerClockConfig': 2, 'TimerClockDivisor': 256, 'CompatibilityOptions': 0, 'VersionInfo': 2, 'DeviceName': 'U3-LV'}
LabJack U3 connected, FIO0-FIO3 initialised


## Scan All FIO Pins (FIO0–FIO7)

Each pin turns ON for 1 second then OFF. Move your LED to each pin to check.

In [ ]:
import time

for pin in range(8):
    val = 1 << pin
    d.getFeedback(u3.PortStateWrite(State=[val, 0, 0], WriteMask=[0xFF, 0, 0]))
    print(f'FIO{pin} ON  (value={val})')
    time.sleep(5.0)
    d.getFeedback(u3.PortStateWrite(State=[0, 0, 0], WriteMask=[0xFF, 0, 0]))
    print(f'FIO{pin} OFF')

print('Scan complete')

FIO0 ON  (value=1)
FIO0 OFF
FIO1 ON  (value=2)
FIO1 OFF
FIO2 ON  (value=4)
FIO2 OFF
FIO3 ON  (value=8)
FIO3 OFF
FIO4 ON  (value=16)
FIO4 OFF
FIO5 ON  (value=32)
FIO5 OFF
FIO6 ON  (value=64)
FIO6 OFF
FIO7 ON  (value=128)
FIO7 OFF
Scan complete


## Manual LED Control

In [2]:
# Turn ON FIO1 (value = 2 = 0b0010)
d.getFeedback(u3.PortStateWrite(State=[2, 0, 0], WriteMask=[0x0F, 0, 0]))
print('FIO1 ON')

FIO1 ON


In [4]:
# Turn OFF FIO1
d.getFeedback(u3.PortStateWrite(State=[0, 0, 0], WriteMask=[0x0F, 0, 0]))
print('FIO1 OFF')

FIO1 OFF


## Send Pulse (simulate experiment trigger)

In [ ]:
import time

def send_pulse(value, pulse_ms=2.0):
    """Send trigger pulse: set value -> wait -> reset to 0."""
    d.getFeedback(u3.PortStateWrite(State=[value, 0, 0], WriteMask=[0x0F, 0, 0]))
    time.sleep(pulse_ms / 1000.0)
    d.getFeedback(u3.PortStateWrite(State=[0, 0, 0], WriteMask=[0x0F, 0, 0]))
    print(f'TRIGGER {value} ({value:04b}) sent')

# Test: send trigger code 2 (FIO1 = maze start)
send_pulse(2)

## Test All Experiment Trigger Codes

In [ ]:
trigger_codes = {
    1:  'Fixation onset',
    2:  'Maze start',
    3:  'Star 1 collected',
    4:  'Star 2 collected',
    5:  'Star 3 collected',
    11: 'Trial complete',
    12: 'Trial escape (ESC)',
}

for code, label in trigger_codes.items():
    send_pulse(code, pulse_ms=2.0)
    print(f'  -> {label}')
    time.sleep(0.5)

## Test via EEGTrigger Class (same as experiment)

In [ ]:
# Close the direct connection first
d.close()
print('Direct connection closed')

## Debug Trigger Test (FIO1 always ON)

In [ ]:
from pywalker.trigger_debug import EEGTriggerDebug, TRIG_FIXATION, TRIG_MAZE_START, TRIG_TRIAL_COMPLETE, TRIG_TRIAL_ESCAPE, star_trigger
import time

trig = EEGTriggerDebug(pulse_ms=500, bits=4, verbose=True)

trig.send(TRIG_FIXATION)        # 1
time.sleep(1.0)
trig.send(TRIG_MAZE_START)      # 2
time.sleep(1.0)
trig.send(star_trigger(0))      # 3 — star 1
time.sleep(1.0)
trig.send(star_trigger(1))      # 4 — star 2
time.sleep(1.0)
trig.send(star_trigger(2))      # 5 — star 3
time.sleep(1.0)
trig.send(TRIG_TRIAL_COMPLETE)  # 11

print('Done')
trig.close()

In [ ]:
from pywalker.trigger import EEGTrigger, TRIG_FIXATION, TRIG_MAZE_START, TRIG_TRIAL_COMPLETE, TRIG_TRIAL_ESCAPE, star_trigger

trig = EEGTrigger(pulse_ms=2.0, bits=4, verbose=True)

trig.send(TRIG_FIXATION)        # 1
time.sleep(0.3)
trig.send(TRIG_MAZE_START)      # 2
time.sleep(0.3)
trig.send(star_trigger(0))      # 3 — star 1
time.sleep(0.3)
trig.send(star_trigger(1))      # 4 — star 2
time.sleep(0.3)
trig.send(star_trigger(2))      # 5 — star 3
time.sleep(0.3)
trig.send(TRIG_TRIAL_COMPLETE)  # 11

print('Sequence done')

In [ ]:
# Close connection
trig.close()
print('LabJack connection closed')